# 数据预处理
## 读取数据集


In [1]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')  # 列名
    f.write('NA,Pave,127500\n')  # 每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')


In [2]:
# 如果没有安装pandas，只需取消对以下行的注释来安装pandas
# !pip install pandas
import pandas as pd

data = pd.read_csv(data_file)
print(data)


   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


## 处理缺失值
注意，“NaN”项代表缺失值。为了处理缺失的数据，典型的方法包括插值法和删除法，其中插值法用一个替代值弥补缺失值，而删除法则直接忽略缺失值。在这里，我们将考虑插值法。

通过位置索引iloc，我们将data分成inputs和outputs，其中前者为data的前两列，而后者为data的最后一列。对于inputs中缺少的数值，我们用同一列的均值替换“NaN”项。

In [4]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
# inputs = inputs.fillna(inputs.mean())
inputs = inputs.fillna(inputs.mean(numeric_only=True))
print(inputs)


   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


对于inputs中的类别值或离散值，我们将“NaN”视为一个类别。由于“巷子类型”（“Alley”）列只接受两种类型的类别值“Pave”和“NaN”， pandas可以自动将此列转换为两列“Alley_Pave”和“Alley_nan”。巷子类型为“Pave”的行会将“Alley_Pave”的值设置为1，“Alley_nan”的值设置为0。缺少巷子类型的行会将“Alley_Pave”和“Alley_nan”分别设置为0和1。

In [5]:
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)


   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


## 转换为张量格式

In [7]:
inputs,outputs

(   NumRooms  Alley_Pave  Alley_nan
 0       3.0        True      False
 1       2.0       False       True
 2       4.0       False       True
 3       3.0       False       True,
 0    127500
 1    106000
 2    178100
 3    140000
 Name: Price, dtype: int64)

In [6]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
X, y


(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

## 小结
- pandas软件包是Python中常用的数据分析工具中，pandas可以与张量兼容。
- 用pandas处理缺失的数据时，我们可根据情况选择用插值法和删除法。
## 练习
创建包含更多行和列的原始数据集。
1. 删除缺失值最多的列。
2. 将预处理后的数据集转换为张量格式。



In [17]:
import os
import pandas as pd

# 创建包含更多行和列的原始数据集。
os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('id,MSSubClass,MSZoning,LotFrontage,NumRooms,Alley,Price\n')  # 列名
    f.write('1,60,RL,65,NA,Pave,127500\n')  # 每行表示一个数据样本
    f.write('2,70,NA,70,2,NA,106000\n')
    f.write('3,NA,NA,80,4,NA,178100\n')
    f.write('4,NA,NA,90,NA,NA,140000\n')
    
original_data = pd.read_csv(data_file)
original_data

,id,MSSubClass,MSZoning,LotFrontage,NumRooms,Alley,Price
0,1,60.0,RL,65,NaN,Pave,127500
1,2,70.0,NaN,70,2.0,NaN,106000
2,3,NaN,NaN,80,4.0,NaN,178100
3,4,NaN,NaN,90,NaN,NaN,140000


**删除缺失值最多的列**

> 解答：

  通过观察，可以看到 _MSZoning_ 和 _Awley_ 列的缺失值一样多，需要考虑到缺失值最多的列有多个的情况使用 `original_data.isna().sum()[original_data.isna().sum() == original_data.isna().sum().max()].index.tolist()` 得到最多缺失值的列名, 再进行删除

In [22]:
original_data.isna().sum()[
    original_data.isna().sum() == original_data.isna().sum().max()
].index.tolist()

['MSZoning', 'Alley']

In [21]:
# data1 = original_data.drop(
#     original_data.isna()
#     .sum()[original_data.isna().sum() == original_data.isna().sum().max()]
#     .index.tolist(),
#     axis=1,
# )
na_counts = original_data.isna().sum()
max_na_cols = na_counts[na_counts == na_counts.max()].index.tolist()
data1 = original_data.drop(columns=max_na_cols)
data1 

,id,MSSubClass,LotFrontage,NumRooms,Price
0,1,60.0,65,NaN,127500
1,2,70.0,70,2.0,106000
2,3,NaN,80,4.0,178100
3,4,NaN,90,NaN,140000


**将预处理后的数据集转换为张量格式**

> 解答：

使用 `torch.tensor` 方法，将数据集转换为张量格式。

In [24]:
# 将Alley列转换成one-hot编码
data2 = pd.get_dummies(data1, dummy_na=True)
data2   

,id,MSSubClass,LotFrontage,NumRooms,Price
0,1,60.0,65,NaN,127500
1,2,70.0,70,2.0,106000
2,3,NaN,80,4.0,178100
3,4,NaN,90,NaN,140000


In [26]:
# 将数据集转换为张量格式
data=torch.tensor(data2.values)
data

tensor([[1.0000e+00, 6.0000e+01, 6.5000e+01,        nan, 1.2750e+05],
        [2.0000e+00, 7.0000e+01, 7.0000e+01, 2.0000e+00, 1.0600e+05],
        [3.0000e+00,        nan, 8.0000e+01, 4.0000e+00, 1.7810e+05],
        [4.0000e+00,        nan, 9.0000e+01,        nan, 1.4000e+05]],
       dtype=torch.float64)